# CMAPSS XGB Anomaly Detection — Sliding Window + Semantic Trends + XGBoost

5-step pipeline:
1. **Load** all 4 CMAPSS datasets (FD001–FD004) with RUL labels
2. **Normalize** operating conditions via KMeans clustering (FD002/FD004 only)
3. **Extract** 30-cycle sliding window features (mean, std, slope, min, max per sensor)
4. **Embed** semantic trend descriptions ("sensor_4 is rapidly increasing") with sentence-transformers
5. **Train** XGBoost on hybrid features (384-dim text embedding + scaled window stats) with class-imbalance handling

In [1]:
!pip install -q numpy pandas scikit-learn matplotlib seaborn tqdm sentence-transformers xgboost

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, precision_recall_curve, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ── Global config ──────────────────────────────────────────────────────────────
SEED = 42
DATA_DIR = "/content/drive/MyDrive/data/"
DATASETS = ["FD001", "FD002", "FD003", "FD004"]
RUL_THRESHOLD = 30
VAL_ENGINE_RATIO = 0.2
MIN_SENSOR_VAR = 1e-6
WINDOW_SIZE = 30
MULTI_CONDITION_DATASETS = {"FD002", "FD004"}  # need operating-condition normalisation
N_CONDITIONS = 6   # KMeans clusters for op-condition grouping
HF_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

XGB_PARAMS = dict(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=-1,
    early_stopping_rounds=40,
    # scale_pos_weight set per-dataset
)

print("Config loaded. Datasets:", DATASETS)

Config loaded. Datasets: ['FD001', 'FD002', 'FD003', 'FD004']


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Core library
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class CmapssDataset:
    name: str
    train: pd.DataFrame
    test: pd.DataFrame
    sensor_cols: List[str]
    op_cols: List[str]


# ── 1. Data loading ────────────────────────────────────────────────────────────

def load_cmapss(dataset="FD001", data_dir=DATA_DIR, rul_threshold=RUL_THRESHOLD,
                min_sensor_var=MIN_SENSOR_VAR) -> CmapssDataset:
    train_df = pd.read_csv(os.path.join(data_dir, f"train_{dataset}.csv")).dropna(axis=1, how="all")
    test_df  = pd.read_csv(os.path.join(data_dir, f"test_{dataset}.csv")).dropna(axis=1, how="all")
    rul_df   = pd.read_csv(os.path.join(data_dir, f"RUL_{dataset}.txt"), header=None, names=["final_rul"])

    # Train RUL
    max_cy = train_df.groupby("unit_number")["time_in_cycles"].max().rename("max_cycle")
    train_df = train_df.merge(max_cy, on="unit_number").copy()
    train_df["RUL"] = train_df["max_cycle"] - train_df["time_in_cycles"]
    train_df.drop(columns=["max_cycle"], inplace=True)

    # Test RUL
    unit_order = np.sort(test_df["unit_number"].unique())
    if len(unit_order) != len(rul_df):
        raise ValueError(f"{dataset}: engine count mismatch with RUL file.")
    final_rul_map = dict(zip(unit_order, rul_df["final_rul"].astype(float)))
    test_df["final_rul"] = test_df["unit_number"].map(final_rul_map)
    max_cy_test = test_df.groupby("unit_number")["time_in_cycles"].transform("max")
    test_df["RUL"] = test_df["final_rul"] + (max_cy_test - test_df["time_in_cycles"])

    # Labels
    train_df["anomaly"] = (train_df["RUL"] <= rul_threshold).astype(int)
    test_df["anomaly"]  = (test_df["RUL"] <= rul_threshold).astype(int)

    op_cols     = [c for c in train_df.columns if c.startswith("operational_setting_")]
    sensor_cols = [c for c in train_df.columns if c.startswith("sensor_measurement_")]
    sensor_cols = [c for c in sensor_cols if train_df[c].var() > min_sensor_var]

    return CmapssDataset(name=dataset, train=train_df, test=test_df,
                         sensor_cols=sensor_cols, op_cols=op_cols)


# ── 2. Operating-condition normalisation (FD002 / FD004) ──────────────────────

def fit_condition_normaliser(df, op_cols, n_clusters=N_CONDITIONS):
    km = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    km.fit(df[op_cols].values)
    scalers = {}
    for c in range(n_clusters):
        mask = km.labels_ == c
        sc = StandardScaler()
        sc.fit(df.loc[mask, df.columns.difference(op_cols + ["unit_number", "time_in_cycles",
                                                              "RUL", "anomaly", "final_rul"])].fillna(0))
        scalers[c] = sc
    return km, scalers


def apply_condition_normaliser(df, km, scalers, op_cols):
    df = df.copy()
    feat_cols = df.columns.difference(op_cols + ["unit_number", "time_in_cycles",
                                                  "RUL", "anomaly", "final_rul"]).tolist()
    labels = km.predict(df[op_cols].values)
    df["_cond"] = labels
    for c, sc in scalers.items():
        mask = df["_cond"] == c
        if mask.sum() == 0:
            continue
        df.loc[mask, feat_cols] = sc.transform(df.loc[mask, feat_cols].fillna(0))
    df.drop(columns=["_cond"], inplace=True)
    return df


def maybe_normalise_conditions(ds: CmapssDataset):
    if ds.name not in MULTI_CONDITION_DATASETS:
        return ds.train.copy(), ds.test.copy()
    print(f"  [{ds.name}] Applying operating-condition normalisation (KMeans k={N_CONDITIONS})")
    km, scalers = fit_condition_normaliser(ds.train, ds.op_cols)
    train_norm = apply_condition_normaliser(ds.train, km, scalers, ds.op_cols)
    test_norm  = apply_condition_normaliser(ds.test,  km, scalers, ds.op_cols)
    return train_norm, test_norm


# ── 3. Sliding-window feature extraction ──────────────────────────────────────

def _linregress_slope(arr: np.ndarray) -> float:
    """Least-squares slope; no scipy needed."""
    n = len(arr)
    if n < 2:
        return 0.0
    x = np.arange(n, dtype=np.float64)
    xm = x.mean(); ym = arr.mean()
    denom = ((x - xm) ** 2).sum()
    if denom == 0:
        return 0.0
    return float(((x - xm) * (arr - ym)).sum() / denom)


def extract_window_features(df: pd.DataFrame, sensor_cols: List[str],
                             window_size: int = WINDOW_SIZE) -> pd.DataFrame:
    """
    For every row, look back up to `window_size` cycles within the same engine
    and compute {sensor}_wmean / wstd / wslope / wmax / wmin.
    Returns a new DataFrame aligned to df.index.
    """
    records = []
    for unit, grp in df.groupby("unit_number", sort=False):
        grp = grp.sort_values("time_in_cycles")
        vals = grp[sensor_cols].values.astype(np.float64)
        n = len(grp)
        rows = []
        for i in range(n):
            lo = max(0, i - window_size + 1)
            win = vals[lo : i + 1]   # shape (w, S)
            row = {}
            for j, col in enumerate(sensor_cols):
                s = win[:, j]
                row[f"{col}_wmean"]  = s.mean()
                row[f"{col}_wstd"]   = s.std() if len(s) > 1 else 0.0
                row[f"{col}_wslope"] = _linregress_slope(s)
                row[f"{col}_wmax"]   = s.max()
                row[f"{col}_wmin"]   = s.min()
            rows.append(row)
        part = pd.DataFrame(rows, index=grp.index)
        records.append(part)
    return pd.concat(records).loc[df.index]


# ── 4. Semantic text generation ───────────────────────────────────────────────

def _slope_phrase(slope: float, std: float, col: str) -> str:
    """Map (slope, std) → human-readable trend phrase."""
    abs_s = abs(slope)
    direction = "increasing" if slope > 0 else "decreasing"
    if abs_s > 0.10:
        speed = "rapidly"
    elif abs_s > 0.02:
        speed = "steadily"
    elif abs_s > 0.005:
        speed = "slowly"
    else:
        return f"{col} is stable (±{std:.3f})"
    return f"{col} is {speed} {direction} ({slope:+.3f}/cycle, variability={std:.3f})"


def window_to_text(slope_row: pd.Series, std_row: pd.Series, sensor_cols: List[str]) -> str:
    phrases = []
    for col in sensor_cols:
        phrases.append(_slope_phrase(slope_row[f"{col}_wslope"], std_row[f"{col}_wstd"], col))
    return "Engine degradation: " + "; ".join(phrases) + "."


def build_texts(win_df: pd.DataFrame, sensor_cols: List[str]) -> List[str]:
    texts = []
    for _, row in win_df.iterrows():
        phrases = []
        for col in sensor_cols:
            phrases.append(_slope_phrase(row[f"{col}_wslope"], row[f"{col}_wstd"], col))
        texts.append("Engine degradation: " + "; ".join(phrases) + ".")
    return texts


print("Core library loaded.")
print(f"  Functions: load_cmapss, maybe_normalise_conditions, extract_window_features,")
print(f"             build_texts, window_to_text")

Core library loaded.
  Functions: load_cmapss, maybe_normalise_conditions, extract_window_features,
             build_texts, window_to_text


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Multi-dataset pipeline driver
# Loads each dataset, normalises conditions, extracts window features,
# builds semantic texts, and stores everything in pipeline_data[dataset].
# ══════════════════════════════════════════════════════════════════════════════

from tqdm.auto import tqdm

pipeline_data: Dict[str, dict] = {}

for DS in DATASETS:
    print(f"\n{'='*60}")
    print(f"  Processing {DS}")
    print(f"{'='*60}")

    # 1. Load
    ds = load_cmapss(DS)
    print(f"  Loaded: train={len(ds.train):,}  test={len(ds.test):,}  sensors={len(ds.sensor_cols)}")

    # 2. Operating-condition normalisation (FD002/FD004 only)
    train_df, test_df = maybe_normalise_conditions(ds)

    # 3. Engine-level train/val split (no data leakage)
    engine_labels = train_df.groupby("unit_number")["anomaly"].max().reset_index()
    try:
        train_units, val_units = train_test_split(
            engine_labels["unit_number"].values,
            test_size=VAL_ENGINE_RATIO,
            random_state=SEED,
            stratify=engine_labels["anomaly"].values,
        )
    except ValueError:
        # FD003/FD004 may have too few anomalous engines for stratify
        train_units, val_units = train_test_split(
            engine_labels["unit_number"].values,
            test_size=VAL_ENGINE_RATIO,
            random_state=SEED,
        )

    train_part = train_df[train_df["unit_number"].isin(train_units)].copy()
    val_part   = train_df[train_df["unit_number"].isin(val_units)].copy()
    test_last  = (test_df.sort_values(["unit_number", "time_in_cycles"])
                         .groupby("unit_number").tail(1).copy())

    print(f"  Split: train={len(train_part):,}  val={len(val_part):,}  "
          f"test_last={len(test_last):,}  test_all={len(test_df):,}")
    print(f"  Anomaly ratio — train: {train_part['anomaly'].mean():.3f}  "
          f"test_last: {test_last['anomaly'].mean():.3f}")

    # 4. Extract sliding-window features
    print(f"  Extracting window features (window={WINDOW_SIZE}) …")
    win_train     = extract_window_features(train_part, ds.sensor_cols)
    win_val       = extract_window_features(val_part,   ds.sensor_cols)
    win_test_all  = extract_window_features(test_df,    ds.sensor_cols)
    # test_last shares same indices as test_df; slice after full extraction
    win_test_last = win_test_all.loc[test_last.index]

    # 5. Build semantic texts
    print(f"  Building semantic texts …")
    texts_train     = build_texts(win_train,     ds.sensor_cols)
    texts_val       = build_texts(win_val,       ds.sensor_cols)
    texts_test_all  = build_texts(win_test_all,  ds.sensor_cols)
    texts_test_last = build_texts(win_test_last, ds.sensor_cols)

    pipeline_data[DS] = dict(
        sensor_cols    = ds.sensor_cols,
        # raw window feature DataFrames
        win_train      = win_train,
        win_val        = win_val,
        win_test_all   = win_test_all,
        win_test_last  = win_test_last,
        # semantic texts
        texts_train     = texts_train,
        texts_val       = texts_val,
        texts_test_all  = texts_test_all,
        texts_test_last = texts_test_last,
        # labels
        y_train    = train_part["anomaly"].astype(int).values,
        y_val      = val_part["anomaly"].astype(int).values,
        y_test_all = test_df["anomaly"].astype(int).values,
        y_test_last= test_last["anomaly"].astype(int).values,
    )

    print(f"  Sample text: {texts_train[0][:120]}…")

print("\nAll datasets processed.")


  Processing FD001
  Loaded: train=20,631  test=13,096  sensors=15
  Split: train=16,340  val=4,291  test_last=100  test_all=13,096
  Anomaly ratio — train: 0.152  test_last: 0.250
  Extracting window features (window=30) …
  Building semantic texts …
  Sample text: Engine degradation: sensor_measurement_2 is stable (±0.000); sensor_measurement_3 is stable (±0.000); sensor_measurement…

  Processing FD002
  Loaded: train=53,759  test=33,991  sensors=21
  [FD002] Applying operating-condition normalisation (KMeans k=6)
  Split: train=42,889  val=10,870  test_last=259  test_all=33,991
  Anomaly ratio — train: 0.150  test_last: 0.236
  Extracting window features (window=30) …
  Building semantic texts …
  Sample text: Engine degradation: sensor_measurement_1 is stable (±0.000); sensor_measurement_2 is stable (±0.000); sensor_measurement…

  Processing FD003
  Loaded: train=24,720  test=16,596  sensors=16
  Split: train=20,177  val=4,543  test_last=100  test_all=16,596
  Anomaly ratio — tr

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Embedding + hybrid feature construction
# Loads SentenceTransformer once, embeds all splits, concatenates with
# StandardScaler-normalised window stats → ~454-dim hybrid feature vectors.
# ══════════════════════════════════════════════════════════════════════════════

from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {HF_MODEL}")
embedder = SentenceTransformer(HF_MODEL)
print("Model loaded.\n")


def encode_texts(texts: List[str], batch_size: int = 256) -> np.ndarray:
    return embedder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,   # L2-normalised → cosine-ready
        convert_to_numpy=True,
    ).astype(np.float32)


def build_hybrid_features(emb: np.ndarray, win_df: pd.DataFrame,
                           scaler: Optional[StandardScaler] = None,
                           fit_scaler: bool = False):
    """
    Concatenate L2-normalised text embedding with StandardScaler-normalised
    raw window statistics.  Returns (hybrid_features, fitted_scaler).
    """
    raw = win_df.values.astype(np.float32)
    if fit_scaler:
        scaler = StandardScaler()
        raw_scaled = scaler.fit_transform(raw)
    else:
        raw_scaled = scaler.transform(raw)
    return np.concatenate([emb, raw_scaled], axis=1), scaler


for DS in DATASETS:
    print(f"\n── {DS} ─────────────────────────────────")
    d = pipeline_data[DS]

    # Embed
    print("  Embedding train …")
    emb_train = encode_texts(d["texts_train"])
    print("  Embedding val …")
    emb_val   = encode_texts(d["texts_val"])
    print("  Embedding test_all …")
    emb_test_all  = encode_texts(d["texts_test_all"])
    print("  Embedding test_last …")
    emb_test_last = encode_texts(d["texts_test_last"])

    # Hybrid features
    X_tr,  scaler = build_hybrid_features(emb_train,     d["win_train"],     fit_scaler=True)
    X_val, _      = build_hybrid_features(emb_val,       d["win_val"],       scaler=scaler)
    X_ta,  _      = build_hybrid_features(emb_test_all,  d["win_test_all"],  scaler=scaler)
    X_tl,  _      = build_hybrid_features(emb_test_last, d["win_test_last"], scaler=scaler)

    print(f"  Hybrid feature dim: {X_tr.shape[1]}  "
          f"(emb={emb_train.shape[1]} + raw={d['win_train'].shape[1]})")

    d.update(dict(X_tr=X_tr, X_val=X_val, X_test_all=X_ta, X_test_last=X_tl, scaler=scaler))

print("\nEmbedding + hybrid feature construction complete.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  Embedding test_all …


Batches:   0%|          | 0/161 [00:00<?, ?it/s]

  Embedding test_last …


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Hybrid feature dim: 489  (emb=384 + raw=105)

Embedding + hybrid feature construction complete.


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# XGBoost training + evaluation + summary table
# ══════════════════════════════════════════════════════════════════════════════

from xgboost import XGBClassifier

# ── Helpers ───────────────────────────────────────────────────────────────────

def best_threshold_f1(y_true: np.ndarray, y_score: np.ndarray):
    """Return (threshold, f1) that maximises F1 on the given split."""
    p, r, th = precision_recall_curve(y_true, y_score)
    f1 = (2 * p * r) / (p + r + 1e-12)
    if len(th) == 0:
        return 0.5, 0.0
    idx = int(np.nanargmax(f1[:-1]))
    return float(th[idx]), float(f1[idx])


def evaluate_split(y_true: np.ndarray, y_pred: np.ndarray,
                   y_score: np.ndarray, label: str) -> dict:
    f1  = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_score)
    except ValueError:
        auc = float("nan")
    cm  = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    print(f"\n  [{label}]  F1={f1:.4f}  AUC={auc:.4f}  TP={tp}  FP={fp}  FN={fn}")
    print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], digits=4))
    return dict(label=label, f1=f1, auc=auc, tp=int(tp), fp=int(fp), fn=int(fn))


# ── Main training loop ────────────────────────────────────────────────────────

summary_rows = []

for DS in DATASETS:
    print(f"\n{'='*65}")
    print(f"  Training XGBoost — {DS}")
    print(f"{'='*65}")

    d = pipeline_data[DS]
    y_tr, y_val = d["y_train"], d["y_val"]

    n_neg = int((y_tr == 0).sum())
    n_pos = int((y_tr == 1).sum())
    spw   = max(1.0, n_neg / max(n_pos, 1))
    print(f"  Class balance: neg={n_neg}  pos={n_pos}  scale_pos_weight={spw:.1f}")

    params = dict(XGB_PARAMS)
    params["scale_pos_weight"] = spw

    clf = XGBClassifier(**params)
    clf.fit(
        d["X_tr"], y_tr,
        eval_set=[(d["X_val"], y_val)],
        verbose=100,
    )

    # Tune threshold on validation set
    val_score = clf.predict_proba(d["X_val"])[:, 1]
    best_th, val_f1 = best_threshold_f1(y_val, val_score)
    print(f"  Best threshold on val: {best_th:.4f}  (val F1={val_f1:.4f})")

    # Evaluate test_last
    score_tl = clf.predict_proba(d["X_test_last"])[:, 1]
    pred_tl  = (score_tl >= best_th).astype(int)
    res_last = evaluate_split(d["y_test_last"], pred_tl, score_tl, f"{DS} test_last")

    # Evaluate test_all
    score_ta = clf.predict_proba(d["X_test_all"])[:, 1]
    pred_ta  = (score_ta >= best_th).astype(int)
    res_all  = evaluate_split(d["y_test_all"],  pred_ta, score_ta, f"{DS} test_all")

    summary_rows.append({
        "Dataset"       : DS,
        "F1 last-cycle" : round(res_last["f1"], 4),
        "AUC last-cycle": round(res_last["auc"], 4),
        "TP/FP/FN last" : f"{res_last['tp']}/{res_last['fp']}/{res_last['fn']}",
        "F1 all-cycles" : round(res_all["f1"], 4),
        "AUC all-cycles": round(res_all["auc"], 4),
    })

    # Store model for potential further analysis
    d["model"]    = clf
    d["threshold"]= best_th

# ── Summary table ─────────────────────────────────────────────────────────────

print("\n" + "="*75)
print("  SUMMARY TABLE")
print("="*75)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print("="*75)

# Highlight FD001 last-cycle target
fd1_f1 = summary_df.loc[summary_df["Dataset"] == "FD001", "F1 last-cycle"].values[0]
if fd1_f1 >= 0.85:
    print(f"\n  FD001 last-cycle F1={fd1_f1:.4f} — target met (>= 0.85)!")
else:
    print(f"\n  FD001 last-cycle F1={fd1_f1:.4f} — below 0.85 target. "
          "Consider larger WINDOW_SIZE or more XGB estimators.")


  Training XGBoost — FD001
  Class balance: neg=13860  pos=2480  scale_pos_weight=5.6
[0]	validation_0-logloss:0.64973
[100]	validation_0-logloss:0.07686
[200]	validation_0-logloss:0.07477
[203]	validation_0-logloss:0.07487
  Best threshold on val: 0.6520  (val F1=0.9152)

  [FD001 test_last]  F1=0.9600  AUC=0.9957  TP=24  FP=1  FN=1
              precision    recall  f1-score   support

      normal     0.9867    0.9867    0.9867        75
     anomaly     0.9600    0.9600    0.9600        25

    accuracy                         0.9800       100
   macro avg     0.9733    0.9733    0.9733       100
weighted avg     0.9800    0.9800    0.9800       100


  [FD001 test_all]  F1=0.8328  AUC=0.9978  TP=274  FP=52  FN=58
              precision    recall  f1-score   support

      normal     0.9955    0.9959    0.9957     12764
     anomaly     0.8405    0.8253    0.8328       332

    accuracy                         0.9916     13096
   macro avg     0.9180    0.9106    0.9143     13096